# Livy · Audio nativo de Gemma 4 para transcribir una clase

**Proyecto:** Livy — asistente de continuidad docente
**Hackday Gemma 4 · GDG CDMX 2026 · Categoría: El futuro de la educación**
**Repositorio:** https://github.com/rvvictor/livy-gemma4-2026-

---

## Por qué existe este notebook

Livy es una app web que convierte cada clase en memoria estructurada y lleva la
bitácora de avance **de cada grupo por separado** contra el plan de estudios.

Al construirla nos topamos con una restricción real de la plataforma:

| Variante de Gemma 4 | Texto | Imagen | Video | **Audio** | Dónde corre |
| :--- | :---: | :---: | :---: | :---: | :--- |
| `gemma-4-31b-it` | ✅ | ✅ | ✅ | ❌ | Gemini API (hospedado) |
| `gemma-4-26b-a4b-it` | ✅ | ✅ | ✅ | ❌ | Gemini API (hospedado) |
| `gemma-4-E2B-it` | ✅ | ✅ | ✅ | **✅** | Autohospedado |
| `gemma-4-E4B-it` | ✅ | ✅ | ✅ | **✅** | Autohospedado |
| `gemma-4-12B-it` | ✅ | ✅ | ✅ | **✅** | Autohospedado |

Las variantes hospedadas gratis en la Gemini API **no aceptan audio**. El audio
nativo vive en E2B, E4B y 12B, que hay que correr uno mismo con GPU.

La laptop con la que se construyó este proyecto es un Ryzen 7 4700U sin GPU
dedicada, así que la app en producción transcribe con la Web Speech API del
navegador y deja el razonamiento a `gemma-4-31b-it`. Fue una decisión de
ingeniería, no de preferencia.

**Este notebook cierra el círculo:** demuestra sobre GPU gratuita que Gemma 4
E2B transcribe una clase real en español *sin ningún modelo externo de ASR*, y
que el mismo modelo convierte esa transcripción en la memoria estructurada que
alimenta la bitácora de Livy.

> Para ejecutarlo: `Settings → Accelerator → GPU T4 x2` y `Internet → On`.

## 1. Dependencias

`transformers` trae el soporte multimodal de Gemma 4; `librosa` y `soundfile`
resuelven la preparación del audio (remuestreo a 16 kHz y conversión a mono).

In [ ]:
!pip install -q -U transformers accelerate librosa soundfile

## 2. Cargar Gemma 4 E2B

Se usa la variante más pequeña con audio nativo. Con 2.3 B de parámetros
efectivos entra sin problemas en una T4 y transcribe a una velocidad razonable.

El pipeline `any-to-any` es el que expone las tres modalidades de entrada:
texto, imagen y audio.

In [ ]:
import torch
from transformers import pipeline

MODELO = "google/gemma-4-E2B-it"

gemma = pipeline(
    task="any-to-any",
    model=MODELO,
    device_map="auto",
    dtype="auto",
)

print("Modelo cargado:", MODELO)
print("Dispositivo:", "GPU" if torch.cuda.is_available() else "CPU")

## 3. Preparar el audio

Gemma 4 espera audio **mono, a 16 kHz, en flotante de 32 bits normalizado a
[-1, 1]**, y acepta **máximo 30 segundos por llamada**. Una clase dura 50 o 90
minutos, así que hay que trocear.

Trocear cada 30 segundos exactos parte palabras a la mitad. En vez de eso
detectamos los silencios naturales del habla con `librosa.effects.split` y
agrupamos los segmentos hasta acercarnos al límite: los cortes caen donde el
profesor respira, no a media palabra.

In [ ]:
import librosa
import numpy as np

FRECUENCIA = 16_000
SEGUNDOS_MAX = 28.0          # margen bajo el límite de 30 s del modelo
UMBRAL_SILENCIO_DB = 32      # cuánto más bajo que el pico cuenta como silencio


def cargar_audio(ruta):
    """Carga cualquier formato y lo deja como Gemma 4 lo necesita."""
    onda, _ = librosa.load(ruta, sr=FRECUENCIA, mono=True)
    return onda.astype(np.float32)


def dividir_en_tramos(onda, segundos_max=SEGUNDOS_MAX):
    """Corta el audio en tramos de <=28 s respetando los silencios del habla."""
    muestras_max = int(segundos_max * FRECUENCIA)
    intervalos = librosa.effects.split(onda, top_db=UMBRAL_SILENCIO_DB)

    tramos, inicio, fin = [], None, None
    for arranque, cierre in intervalos:
        if inicio is None:
            inicio, fin = arranque, cierre
            continue
        if cierre - inicio <= muestras_max:
            fin = cierre                      # cabe: se extiende el tramo actual
        else:
            tramos.append(onda[inicio:fin])
            inicio, fin = arranque, cierre
    if inicio is not None:
        tramos.append(onda[inicio:fin])

    # Un bloque de habla continua puede exceder por sí solo el límite: se parte.
    finales = []
    for tramo in tramos:
        if len(tramo) <= muestras_max:
            finales.append(tramo)
        else:
            for i in range(0, len(tramo), muestras_max):
                finales.append(tramo[i:i + muestras_max])
    return [t for t in finales if len(t) > FRECUENCIA // 2]   # se descarta <0.5 s

### Audio de prueba

Sube tu propio archivo con el panel **Input** de Kaggle y ajusta `RUTA_AUDIO`.

Si no tienes uno a la mano, la celda siguiente sintetiza un tono de prueba solo
para que el notebook corra de principio a fin. Para la evaluación real usa una
grabación de voz: el valor de esta demostración está en el habla.

In [ ]:
from pathlib import Path
import soundfile as sf

RUTA_AUDIO = "clase.wav"   # <-- reemplaza por tu grabación

if not Path(RUTA_AUDIO).exists():
    print("No se encontró", RUTA_AUDIO, "— generando un audio de prueba.")
    duracion = 40
    t = np.linspace(0, duracion, duracion * FRECUENCIA, endpoint=False)
    señal = 0.2 * np.sin(2 * np.pi * 220 * t) * (np.sin(2 * np.pi * 0.5 * t) > 0)
    sf.write(RUTA_AUDIO, señal.astype(np.float32), FRECUENCIA)

onda = cargar_audio(RUTA_AUDIO)
tramos = dividir_en_tramos(onda)

print(f"Duración total: {len(onda) / FRECUENCIA:.1f} s")
print(f"Tramos generados: {len(tramos)}")
for i, tramo in enumerate(tramos[:5], 1):
    print(f"  tramo {i}: {len(tramo) / FRECUENCIA:.1f} s")

## 4. Transcribir con el audio nativo de Gemma 4

El prompt sigue la recomendación de la documentación de Gemma para ASR: pedir
la transcripción y **nada más**, indicando el idioma de forma explícita. Sin esa
instrucción el modelo tiende a resumir o a comentar el audio en vez de
transcribirlo literalmente.

In [ ]:
PROMPT_ASR = (
    "Transcribe literalmente el siguiente fragmento de una clase impartida en "
    "español de México. Devuelve únicamente la transcripción, sin comentarios, "
    "sin comillas y sin saltos de línea."
)


def transcribir_tramo(tramo):
    mensajes = [{
        "role": "user",
        "content": [
            {"type": "text", "text": PROMPT_ASR},
            {"type": "audio", "audio": tramo},
        ],
    }]
    salida = gemma(
        mensajes,
        return_full_text=False,
        generate_kwargs={"max_new_tokens": 512, "do_sample": False},
    )
    return salida[0]["generated_text"].strip()


partes = []
for indice, tramo in enumerate(tramos, start=1):
    texto = transcribir_tramo(tramo)
    partes.append(texto)
    print(f"[{indice}/{len(tramos)}] {texto[:100]}...")

transcripcion = " ".join(partes)
print("\n--- TRANSCRIPCIÓN COMPLETA ---\n")
print(transcripcion)

## 5. De transcripción a memoria estructurada

Aquí es donde Livy deja de ser un transcriptor y se vuelve un asistente de
continuidad. **El mismo modelo** recibe la transcripción junto con el plan de
estudios y el punto donde venía ese grupo, y devuelve:

- el resumen de la sesión,
- los puntos clave,
- **el mapeo de qué temas del temario se tocaron y con qué profundidad**.

Ese último campo es el que hace avanzar la bitácora de la sección. En la app
este paso corre sobre `gemma-4-31b-it` hospedado; aquí lo hace E2B para que el
notebook sea autocontenido.

In [ ]:
import json

TEMARIO = [
    {"id": 5, "titulo": "Álgebra de funciones y composición"},
    {"id": 6, "titulo": "Noción intuitiva de límite"},
    {"id": 7, "titulo": "Cálculo de límites"},
    {"id": 8, "titulo": "Límites laterales e infinitos"},
]

CONTEXTO_PREVIO = (
    "El grupo 3CV3 lleva 6 sesiones y un avance del 53% del plan. "
    "El último tema cubierto fue [6] Noción intuitiva de límite. "
    "El tema [7] Cálculo de límites quedó solo introducido porque la sesión "
    "se interrumpió por un simulacro."
)

prompt_memoria = f"""Acaba de terminar una sesión de Cálculo Diferencial con el grupo 3CV3.

Plan de estudios (usa estos identificadores tal cual):
{json.dumps(TEMARIO, ensure_ascii=False, indent=2)}

Dónde venía este grupo:
{CONTEXTO_PREVIO}

Transcripción de la sesión:
---
{transcripcion}
---

Devuelve únicamente JSON con esta forma:
{{"titulo": "...", "resumen": "...", "puntos_clave": ["..."],
  "donde_quedo": "...",
  "temas": [{{"tema_id": 7, "nivel": "cubierto", "evidencia": "cita breve"}}]}}

Reglas: solo temas del plan de arriba; "nivel" es introducido, cubierto o
reforzado; sin cita literal de la transcripción no marques el tema."""

salida = gemma(
    [{"role": "user", "content": [{"type": "text", "text": prompt_memoria}]}],
    return_full_text=False,
    generate_kwargs={"max_new_tokens": 1024, "do_sample": False},
)

crudo = salida[0]["generated_text"].strip()
if crudo.startswith("```"):
    crudo = crudo.split("\n", 1)[-1].rsplit("```", 1)[0]

try:
    memoria = json.loads(crudo)
    print(json.dumps(memoria, ensure_ascii=False, indent=2))
except json.JSONDecodeError:
    print("Salida cruda del modelo:\n", crudo)

## 6. Qué demuestra este notebook

1. **Gemma 4 transcribe español sin ASR externo.** El audio entra directo al
   modelo; no hay Whisper ni ningún otro reconocedor en el camino.
2. **El límite de 30 segundos es manejable.** Cortar en los silencios naturales
   del habla, en vez de a intervalos fijos, evita partir palabras.
3. **Un solo modelo cubre el pipeline completo** de Livy: audio → texto →
   memoria estructurada → mapeo contra el plan de estudios.

### Limitaciones que reconocemos

- El troceo pierde el contexto entre tramos: el modelo no sabe qué se dijo en el
  tramo anterior. Para una clase completa habría que pasar las últimas frases
  como contexto, a costa de más tokens.
- E2B es la variante más pequeña. Con E4B o 12B la transcripción mejora de forma
  notoria, a cambio de más memoria de GPU.
- En la app en producción este paso lo hace el navegador precisamente porque la
  GPU no está garantizada del lado del usuario. Este notebook demuestra que la
  ruta 100% Gemma existe y funciona; la app elige la que no se cae en vivo.